# Text Mining & Image Recognition — Laboratorio #1

**Postgrado en Análisis y Predicción de Datos — Tercer Trimestre, 2026**  
**Curso:** IIO  
**Estudiante:** Robinson López Hidalgo  
**Carnet:** 24009443

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import os

# Helper: mostrar imagen en notebook (convierte BGR→RGB para matplotlib)
def mostrar(img, titulo='', cmap=None):
    plt.figure(figsize=(7, 5))
    if cmap:
        plt.imshow(img, cmap=cmap)
    else:
        plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(titulo, fontsize=11)
    plt.axis('off')
    plt.tight_layout()
    plt.show()

---
## Problema 1 — Función de filtrado de canales de color

La función recibe una imagen y un entero `color` y devuelve la imagen con únicamente los canales indicados activos. Los demás canales se ponen en cero.

In [ ]:
def filtrar_canal(imagen, color):
    """
    Recibe una imagen BGR y un entero color.
    Devuelve la imagen con los canales activos según:
      1  -> solo azul
      2  -> solo verde
      3  -> solo rojo
      10 -> rojo y verde
      20 -> verde y azul
      30 -> azul y rojo
    """
    resultado = np.zeros_like(imagen)

    # OpenCV usa orden B=0, G=1, R=2
    if color == 1:       # solo azul
        resultado[:, :, 0] = imagen[:, :, 0]
    elif color == 2:     # solo verde
        resultado[:, :, 1] = imagen[:, :, 1]
    elif color == 3:     # solo rojo
        resultado[:, :, 2] = imagen[:, :, 2]
    elif color == 10:    # rojo y verde
        resultado[:, :, 1] = imagen[:, :, 1]
        resultado[:, :, 2] = imagen[:, :, 2]
    elif color == 20:    # verde y azul
        resultado[:, :, 0] = imagen[:, :, 0]
        resultado[:, :, 1] = imagen[:, :, 1]
    elif color == 30:    # azul y rojo
        resultado[:, :, 0] = imagen[:, :, 0]
        resultado[:, :, 2] = imagen[:, :, 2]
    else:
        raise ValueError(f'Valor de color no reconocido: {color}')

    return resultado

In [ ]:
# Prueba con una imagen de naturaleza
img_prueba = cv2.imread('imagen1_reconstruida.jpg')

configs = [
    (1,  'color=1: Solo Azul'),
    (2,  'color=2: Solo Verde'),
    (3,  'color=3: Solo Rojo'),
    (10, 'color=10: Rojo + Verde'),
    (20, 'color=20: Verde + Azul'),
    (30, 'color=30: Azul + Rojo'),
]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('Problema 1 — Filtrado de canales de color', fontsize=13, fontweight='bold')

for ax, (c, titulo) in zip(axes.flatten(), configs):
    resultado = filtrar_canal(img_prueba, c)
    ax.imshow(cv2.cvtColor(resultado, cv2.COLOR_BGR2RGB))
    ax.set_title(titulo, fontsize=9)
    ax.axis('off')

plt.tight_layout()
plt.savefig('p1_filtrado_canales.png', bbox_inches='tight', dpi=110)
plt.show()
print('Problema 1 completado.')

---
## Problema 2 — Reconstrucción de imagen a color desde grises 3D

Las imágenes en escala de grises 3D fueron generadas asignando cada canal de color (R, G, B) de la imagen original a una imagen en escala de grises independiente (donde los tres canales de cada imagen tienen el mismo valor). Para reconstruir la imagen original se extrae el canal 0 de cada imagen gris y se usan como B, G y R respectivamente.

In [ ]:
def reconstruir_color(img_gray_rojo, img_gray_verde, img_gray_azul):
    """
    Recibe tres imágenes en escala de grises 3D, cada una representando
    el canal R, G y B de la imagen original respectivamente.
    Devuelve la imagen original a color en formato BGR.
    """
    canal_r = img_gray_rojo[:, :, 0]   # los 3 canales son iguales; tomamos el 0
    canal_g = img_gray_verde[:, :, 0]
    canal_b = img_gray_azul[:, :, 0]

    # cv2.merge espera orden BGR
    imagen_color = cv2.merge([canal_b, canal_g, canal_r])
    return imagen_color

In [ ]:
# Reconstrucción para los tres conjuntos de imágenes
nombres = ['imagen1', 'imagen2', 'perro']

fig, axes = plt.subplots(3, 4, figsize=(16, 11))
fig.suptitle('Problema 2 — Reconstrucción de imagen a color desde grises 3D', 
             fontsize=12, fontweight='bold')

for i, nombre in enumerate(nombres):
    img_r = cv2.imread(f'{nombre}_salida_gray_rojo.jpg')
    img_g = cv2.imread(f'{nombre}_salida_gray_verde.jpg')
    img_b = cv2.imread(f'{nombre}_salida_gray_azul.jpg')

    reconstruida = reconstruir_color(img_r, img_g, img_b)
    cv2.imwrite(f'{nombre}_reconstruida.jpg', reconstruida)

    axes[i, 0].imshow(img_r[:,:,0], cmap='gray')
    axes[i, 0].set_title(f'{nombre} — Gray Rojo', fontsize=8); axes[i, 0].axis('off')

    axes[i, 1].imshow(img_g[:,:,0], cmap='gray')
    axes[i, 1].set_title(f'{nombre} — Gray Verde', fontsize=8); axes[i, 1].axis('off')

    axes[i, 2].imshow(img_b[:,:,0], cmap='gray')
    axes[i, 2].set_title(f'{nombre} — Gray Azul', fontsize=8); axes[i, 2].axis('off')

    axes[i, 3].imshow(cv2.cvtColor(reconstruida, cv2.COLOR_BGR2RGB))
    axes[i, 3].set_title(f'{nombre} — RECONSTRUIDA', fontsize=8, fontweight='bold')
    axes[i, 3].axis('off')

plt.tight_layout()
plt.savefig('p2_reconstruccion.png', bbox_inches='tight', dpi=110)
plt.show()
print('Problema 2 completado. Imágenes reconstruidas guardadas.')

---
## Problema 3 — Convertir imagen a escala de grises 3D

La función recibe una imagen a color y genera tres imágenes de salida en escala de grises 3D (una por canal). Cada imagen de salida es una imagen donde los tres canales contienen el mismo valor: el del canal original correspondiente. Se usa una imagen diferente a las del Problema 2.

In [ ]:
def convertir_a_gray_3d(imagen_color):
    """
    Recibe una imagen BGR a color.
    Devuelve tres imágenes en escala de grises 3D:
      - gray_r: canal R replicado en los 3 canales
      - gray_g: canal G replicado en los 3 canales
      - gray_b: canal B replicado en los 3 canales
    """
    canal_b = imagen_color[:, :, 0]
    canal_g = imagen_color[:, :, 1]
    canal_r = imagen_color[:, :, 2]

    # Replicar cada canal en los 3 canales para crear imagen 3D en grises
    gray_r = cv2.merge([canal_r, canal_r, canal_r])
    gray_g = cv2.merge([canal_g, canal_g, canal_g])
    gray_b = cv2.merge([canal_b, canal_b, canal_b])

    return gray_r, gray_g, gray_b

In [ ]:
# Imagen de prueba diferente a las del Problema 2
# Usamos imagen2_reconstruida que ya generamos en el Problema 2
img_test_p3 = cv2.imread('imagen2_reconstruida.jpg')

gray_r, gray_g, gray_b = convertir_a_gray_3d(img_test_p3)

# Guardar resultados
cv2.imwrite('p3_prueba_gray_rojo.jpg', gray_r)
cv2.imwrite('p3_prueba_gray_verde.jpg', gray_g)
cv2.imwrite('p3_prueba_gray_azul.jpg', gray_b)

# Verificar: los canales de cada imagen gris deben ser iguales
print('Verificación canal R — canales iguales:', 
      np.allclose(gray_r[:,:,0], gray_r[:,:,1]) and np.allclose(gray_r[:,:,0], gray_r[:,:,2]))
print('Verificación canal G — canales iguales:', 
      np.allclose(gray_g[:,:,0], gray_g[:,:,1]) and np.allclose(gray_g[:,:,0], gray_g[:,:,2]))
print('Verificación canal B — canales iguales:', 
      np.allclose(gray_b[:,:,0], gray_b[:,:,1]) and np.allclose(gray_b[:,:,0], gray_b[:,:,2]))

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
fig.suptitle('Problema 3 — Imagen a escala de grises 3D', fontsize=12, fontweight='bold')

axes[0].imshow(cv2.cvtColor(img_test_p3, cv2.COLOR_BGR2RGB))
axes[0].set_title('Original', fontsize=9); axes[0].axis('off')

axes[1].imshow(gray_r[:,:,0], cmap='gray')
axes[1].set_title('Gray 3D — Canal Rojo', fontsize=9); axes[1].axis('off')

axes[2].imshow(gray_g[:,:,0], cmap='gray')
axes[2].set_title('Gray 3D — Canal Verde', fontsize=9); axes[2].axis('off')

axes[3].imshow(gray_b[:,:,0], cmap='gray')
axes[3].set_title('Gray 3D — Canal Azul', fontsize=9); axes[3].axis('off')

plt.tight_layout()
plt.savefig('p3_gray3d.png', bbox_inches='tight', dpi=110)
plt.show()
print('Problema 3 completado.')

---
## Problema 4 — Histogramas de canales de color y escala de grises

La función muestra el histograma de cada canal de color (R, G, B) y el de escala de grises calculado como promedio aritmético de los tres canales, sin usar funciones de OpenCV para la escala de grises. Cada histograma incluye una línea vertical con la media de la distribución.

In [ ]:
def histogramas_canales(imagen, titulo_base='Imagen'):
    """
    Recibe una imagen BGR.
    Muestra 4 histogramas: B, G, R y escala de grises (promedio aritmético).
    Cada histograma incluye una línea vertical con la media de la distribución.
    No usa funciones de OpenCV para la escala de grises.
    """
    canal_b = imagen[:, :, 0].flatten().astype(np.float64)
    canal_g = imagen[:, :, 1].flatten().astype(np.float64)
    canal_r = imagen[:, :, 2].flatten().astype(np.float64)

    # Escala de grises como promedio aritmético (sin OpenCV)
    gray = ((imagen[:, :, 0].astype(np.float64) +
             imagen[:, :, 1].astype(np.float64) +
             imagen[:, :, 2].astype(np.float64)) / 3.0).flatten()

    canales = [
        (canal_b, 'blue',  'Canal Azul (B)'),
        (canal_g, 'green', 'Canal Verde (G)'),
        (canal_r, 'red',   'Canal Rojo (R)'),
        (gray,    'gray',  'Escala de Grises (promedio aritmético)'),
    ]

    fig, axes = plt.subplots(1, 4, figsize=(18, 4))
    fig.suptitle(f'Problema 4 — Histogramas: {titulo_base}', 
                 fontsize=12, fontweight='bold')

    for ax, (datos, color, nombre) in zip(axes, canales):
        media = np.mean(datos)
        ax.hist(datos, bins=256, range=(0, 255), color=color, alpha=0.75, density=True)
        ax.axvline(media, color='black', linewidth=2, linestyle='--',
                   label=f'Media = {media:.1f}')
        ax.set_title(nombre, fontsize=9)
        ax.set_xlabel('Valor de píxel (0-255)')
        ax.set_ylabel('Densidad')
        ax.legend(fontsize=8)
        ax.set_xlim(0, 255)

    plt.tight_layout()
    plt.savefig(f'p4_histogramas_{titulo_base.replace(" ","_")}.png',
                bbox_inches='tight', dpi=110)
    plt.show()

In [ ]:
# Aplicar a las tres imágenes reconstruidas
for nombre in ['imagen1', 'imagen2', 'perro']:
    img = cv2.imread(f'{nombre}_reconstruida.jpg')
    histogramas_canales(img, titulo_base=nombre)

print('Problema 4 completado.')

---
## Problema 5 — Escala de grises ponderada

### ¿En qué consiste la escala de grises ponderada?

El enfoque estándar (promedio aritmético) trata los tres canales de color por igual: `gray = (R + G + B) / 3`. Sin embargo, el ojo humano no percibe los colores con la misma sensibilidad. Somos más sensibles al verde, moderadamente al rojo y mucho menos al azul.

La escala de grises ponderada asigna un peso diferente a cada canal según la percepción humana:

$$\text{gray} = w_R \cdot R + w_G \cdot G + w_B \cdot B \quad \text{con} \quad w_R + w_G + w_B = 1$$

Los pesos más usados son los del estándar **ITU-R BT.601** (televisión analógica):
- $w_R = 0.299$
- $w_G = 0.587$  
- $w_B = 0.114$

Y los del estándar **ITU-R BT.709** (HDTV, pantallas modernas):
- $w_R = 0.2126$
- $w_G = 0.7152$
- $w_B = 0.0722$

No existe una solución única porque los pesos óptimos dependen del espacio de color del monitor y del propósito de la aplicación.

In [ ]:
def gray_ponderado(imagen, wr=0.299, wg=0.587, wb=0.114):
    """
    Convierte una imagen BGR a escala de grises ponderada.
    Pesos por defecto: estándar ITU-R BT.601
      wr=0.299 (rojo), wg=0.587 (verde), wb=0.114 (azul)
    Retorna imagen en escala de grises (2D, uint8).
    """
    if abs(wr + wg + wb - 1.0) > 1e-6:
        raise ValueError(f'Los pesos deben sumar 1. Suma actual: {wr+wg+wb}')

    b = imagen[:, :, 0].astype(np.float64)
    g = imagen[:, :, 1].astype(np.float64)
    r = imagen[:, :, 2].astype(np.float64)

    gray = wr * r + wg * g + wb * b
    return np.clip(gray, 0, 255).astype(np.uint8)

In [ ]:
img_p5 = cv2.imread('imagen1_reconstruida.jpg')

# Comparar diferentes esquemas de ponderación
gray_promedio  = ((img_p5[:,:,0].astype(np.float64) +
                   img_p5[:,:,1].astype(np.float64) +
                   img_p5[:,:,2].astype(np.float64)) / 3).astype(np.uint8)

gray_bt601 = gray_ponderado(img_p5, wr=0.299,  wg=0.587,  wb=0.114)
gray_bt709 = gray_ponderado(img_p5, wr=0.2126, wg=0.7152, wb=0.0722)
gray_custom= gray_ponderado(img_p5, wr=0.45,   wg=0.40,   wb=0.15)  # ponderación propia

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
fig.suptitle('Problema 5 — Comparación de escalas de grises ponderadas', 
             fontsize=11, fontweight='bold')

for ax, (img_g, titulo) in zip(axes, [
    (gray_promedio, 'Promedio aritmético\n(1/3, 1/3, 1/3)'),
    (gray_bt601,    'BT.601\n(R=0.299, G=0.587, B=0.114)'),
    (gray_bt709,    'BT.709\n(R=0.2126, G=0.7152, B=0.0722)'),
    (gray_custom,   'Personalizado\n(R=0.45, G=0.40, B=0.15)'),
]):
    ax.imshow(img_g, cmap='gray')
    ax.set_title(titulo, fontsize=8)
    ax.axis('off')

plt.tight_layout()
plt.savefig('p5_gray_ponderado.png', bbox_inches='tight', dpi=110)
plt.show()
print('Problema 5 completado.')

---
## Problema 6 — Espacio de color HSV

### ¿Qué es el espacio de color HSV?

HSV son las siglas de **Hue (Matiz), Saturation (Saturación) y Value (Valor/Brillo)**. Es una representación alternativa del espacio de color RGB diseñada para ser más intuitiva desde la perspectiva humana de percepción del color.

### Componentes

| Componente | Descripción | Rango (OpenCV) |
|---|---|---|
| **H — Hue (Matiz)** | El tono o tipo de color. Representa la posición en el círculo cromático: 0°=rojo, 60°=amarillo, 120°=verde, 180°=cian, 240°=azul, 300°=magenta. | 0 a 179 (OpenCV divide el ángulo a la mitad) |
| **S — Saturation (Saturación)** | La pureza o intensidad del color. 0 = gris neutro (sin color), 255 = color completamente puro. | 0 a 255 |
| **V — Value (Valor/Brillo)** | La luminosidad del color. 0 = negro, 255 = máximo brillo. | 0 a 255 |

### ¿Cómo se mapean los colores desde RGB a HSV?

Dado un píxel RGB normalizado a $[0,1]$, se define:

$$C_{\max} = \max(R,G,B) \qquad C_{\min} = \min(R,G,B) \qquad \Delta = C_{\max} - C_{\min}$$

**Valor (V):**
$$V = C_{\max}$$

**Saturación (S):**
$$S = \begin{cases} 0 & \text{si } C_{\max} = 0 \\ \dfrac{\Delta}{C_{\max}} & \text{si } C_{\max} \neq 0 \end{cases}$$

**Matiz (H):**
$$H = \begin{cases} 0 & \text{si } \Delta = 0 \\ 60 \times \left(\dfrac{G-B}{\Delta} \bmod 6\right) & \text{si } C_{\max} = R \\ 60 \times \left(\dfrac{B-R}{\Delta} + 2\right) & \text{si } C_{\max} = G \\ 60 \times \left(\dfrac{R-G}{\Delta} + 4\right) & \text{si } C_{\max} = B \end{cases}$$

### ¿Por qué es útil HSV?

En RGB es difícil aislar un color específico porque el tono, la saturación y el brillo están mezclados en los tres canales. En HSV se pueden filtrar colores por rango de matiz (H) independientemente de la iluminación (V), lo que lo hace ideal para segmentación de objetos por color, detección de piel, seguimiento visual y procesamiento de imágenes médicas.

Por ejemplo, para detectar todos los objetos rojos de una imagen, en HSV basta con filtrar H ∈ [0, 10] ∪ [170, 179], mientras que en RGB sería necesario definir rangos en los tres canales simultáneamente.

In [ ]:
# Visualización del espacio HSV en los canales de la imagen
img_hsv_demo = cv2.imread('perro_reconstruida.jpg')
hsv = cv2.cvtColor(img_hsv_demo, cv2.COLOR_BGR2HSV)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
fig.suptitle('Problema 6 — Espacio de color HSV', fontsize=12, fontweight='bold')

axes[0].imshow(cv2.cvtColor(img_hsv_demo, cv2.COLOR_BGR2RGB))
axes[0].set_title('Original (RGB)', fontsize=9); axes[0].axis('off')

axes[1].imshow(hsv[:,:,0], cmap='hsv')
axes[1].set_title('H — Matiz (Hue)', fontsize=9); axes[1].axis('off')

axes[2].imshow(hsv[:,:,1], cmap='gray')
axes[2].set_title('S — Saturación', fontsize=9); axes[2].axis('off')

axes[3].imshow(hsv[:,:,2], cmap='gray')
axes[3].set_title('V — Valor/Brillo', fontsize=9); axes[3].axis('off')

plt.tight_layout()
plt.savefig('p6_espacio_hsv.png', bbox_inches='tight', dpi=110)
plt.show()
print('Problema 6 completado.')